<a href="https://colab.research.google.com/github/NicKylis/SveltNet/blob/development/noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, accuracy_score
import os
import requests
import zipfile
import io
import gzip
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Create directory and download EMNIST dataset
os.makedirs('res', exist_ok=True)
data = requests.get('https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip')
files = zipfile.ZipFile(io.BytesIO(data.content))
files.extractall('res')

# Validation split & batch size
validation_split = 0.3
batch_size = 512
num_of_classes = 63  # 62 EMNIST classes + 1 FashionMNIST class

# Function to read MNIST images (for EMNIST)
def read_MNIST_images(filename):
    with gzip.open(filename, 'rb') as file:
        images = np.frombuffer(file.read(), np.uint8, offset=16)
    return images.reshape(-1, 28, 28).astype("float32") / 255.0

# Function to read MNIST labels (for EMNIST)
def read_MNIST_labels(filename):
    with gzip.open(filename, 'rb') as file:
        labels = np.frombuffer(file.read(), np.uint8, offset=8)
    return labels

# Load EMNIST data
x_train = read_MNIST_images('res/gzip/emnist-byclass-train-images-idx3-ubyte.gz')
y_train = read_MNIST_labels('res/gzip/emnist-byclass-train-labels-idx1-ubyte.gz')
x_test = read_MNIST_images('res/gzip/emnist-byclass-test-images-idx3-ubyte.gz')
y_test = read_MNIST_labels('res/gzip/emnist-byclass-test-labels-idx1-ubyte.gz')

# Split EMNIST into train & validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=validation_split, random_state=42)

# Load FashionMNIST data
transform = transforms.Compose([transforms.ToTensor()])
fashion_train_dataset = datasets.FashionMNIST(root='res', train=True, download=True, transform=transform)
fashion_test_dataset = datasets.FashionMNIST(root='res', train=False, download=True, transform=transform)

fashion_train_images = fashion_train_dataset.data.numpy().astype("float32") / 255.0  # Shape: (60000, 28, 28)
fashion_train_labels = fashion_train_dataset.targets.numpy()  # Shape: (60000,)
fashion_test_images = fashion_test_dataset.data.numpy().astype("float32") / 255.0   # Shape: (10000, 28, 28)
fashion_test_labels = fashion_test_dataset.targets.numpy()  # Shape: (10000,)

# Assign FashionMNIST samples label 62 (new class)
fashion_train_labels = np.full_like(fashion_train_labels, 62)
fashion_test_labels = np.full_like(fashion_test_labels, 62)

# Select a subset of FashionMNIST (e.g., 10% of each split)
num_fashion_train = len(fashion_train_images) // 10  # 6000 samples
num_fashion_test = len(fashion_test_images) // 10    # 1000 samples

indices_train = np.random.choice(len(fashion_train_images), num_fashion_train, replace=False)
indices_test = np.random.choice(len(fashion_test_images), num_fashion_test, replace=False)

fashion_train_images_subset = fashion_train_images[indices_train]
fashion_train_labels_subset = fashion_train_labels[indices_train]
fashion_test_images_subset = fashion_test_images[indices_test]
fashion_test_labels_subset = fashion_test_labels[indices_test]

# Split FashionMNIST training subset into train and validation
fashion_train_subset, fashion_val_subset, fashion_train_labels_subset, fashion_val_labels_subset = train_test_split(
    fashion_train_images_subset,
    fashion_train_labels_subset,
    test_size=validation_split,
    random_state=42
)

# Combine EMNIST and FashionMNIST
x_train_combined = np.concatenate([x_train, fashion_train_subset], axis=0)
y_train_combined = np.concatenate([y_train, fashion_train_labels_subset], axis=0)
x_val_combined = np.concatenate([x_val, fashion_val_subset], axis=0)
y_val_combined = np.concatenate([y_val, fashion_val_labels_subset], axis=0)
x_test_combined = np.concatenate([x_test, fashion_test_images_subset], axis=0)
y_test_combined = np.concatenate([y_test, fashion_test_labels_subset], axis=0)

# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train_combined, dtype=torch.float32).unsqueeze(1)  # (N, 1, H, W)
y_train_tensor = torch.tensor(y_train_combined, dtype=torch.long)
x_val_tensor = torch.tensor(x_val_combined, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val_combined, dtype=torch.long)
x_test_tensor = torch.tensor(x_test_combined, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test_combined, dtype=torch.long)

# PyTorch Dataset class
class MNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Create datasets
train_dataset = MNISTDataset(x_train_tensor, y_train_tensor)
val_dataset = MNISTDataset(x_val_tensor, y_val_tensor)
test_dataset = MNISTDataset(x_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the CNN model
class CNNModel(nn.Module):
    def __init__(self, num_of_classes=63):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(16)

        self.dwconv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.15)

        self.dwconv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn5 = nn.BatchNorm2d(32)

        self.dwconv6 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn6 = nn.BatchNorm2d(32)

        self.conv7 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn7 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.2)

        self.dwconv8 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn8 = nn.BatchNorm2d(64)

        self.dwconv9 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn9 = nn.BatchNorm2d(64)

        self.conv10 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn10 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.2)

        self.dwconv11 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn11 = nn.BatchNorm2d(64)

        self.dwconv12 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn12 = nn.BatchNorm2d(64)

        self.conv13 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn13 = nn.BatchNorm2d(64)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop4 = nn.Dropout(0.2)

        self.dwconv14 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn14 = nn.BatchNorm2d(64)

        self.dwconv15 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn15 = nn.BatchNorm2d(64)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.drop5 = nn.Dropout(0.25)
        self.fc = nn.Linear(64, num_of_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.dwconv2(x)))
        x = F.relu(self.bn3(self.dwconv3(x)))

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = F.relu(self.bn5(self.dwconv5(x)))
        x = F.relu(self.bn6(self.dwconv6(x)))

        x = F.relu(self.bn7(self.conv7(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = F.relu(self.bn8(self.dwconv8(x)))
        x = F.relu(self.bn9(self.dwconv9(x)))

        x = F.relu(self.bn10(self.conv10(x)))
        x = self.pool3(x)
        x = self.drop3(x)

        x = F.relu(self.bn11(self.dwconv11(x)))
        x = F.relu(self.bn12(self.dwconv12(x)))

        x = F.relu(self.bn13(self.conv13(x)))
        x = self.pool4(x)
        x = self.drop4(x)

        x = F.relu(self.bn14(self.dwconv14(x)))
        x = F.relu(self.bn15(self.dwconv15(x)))

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.drop5(x)
        x = self.fc(x)
        return x

# Initialize model
model = CNNModel(num_of_classes=num_of_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=2e-3)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=2.5e-5)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_weights = model.state_dict() if self.restore_best_weights else None
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                if self.restore_best_weights and self.best_model_weights is not None:
                    model.load_state_dict(self.best_model_weights)
                return True
        return False

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

# Training Loop
num_epochs = 30
best_val_loss = float('inf')
best_model_path = 'best_emnist_fashion_model_state_dict.pth'

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = torch.nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= len(train_loader.dataset)
    train_accuracy = 100. * train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = torch.nn.CrossEntropyLoss()(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_accuracy = 100. * val_correct / val_total

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

    scheduler.step(val_loss)

    if early_stopping(val_loss, model):
        break

    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

# Save the final model
final_model_path = 'final_emnist_fashion_model_state_dict.pth'
torch.save(model.state_dict(), final_model_path)
print(f"Saved final model to {final_model_path}")

# Test Phase
model.eval()
all_preds = []
all_labels = []
test_loss = 0.0
test_correct = 0
test_total = 0

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_loss /= len(test_loader.dataset)
test_accuracy = 100. * test_correct / test_total
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Test Precision (Weighted): {precision:.4f}")
print(f"Test Recall (Weighted): {recall:.4f}")

# Per-class precision and recall for FashionMNIST (class 62)
precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0)
print(f"Class 62 (FashionMNIST) Precision: {precision_per_class[62]:.4f}")
print(f"Class 62 (FashionMNIST) Recall: {recall_per_class[62]:.4f}")

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.96MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.9MB/s]


Saved best model with Val Loss: 0.4754
Epoch 1/60 - Train Loss: 1.0941, Train Acc: 69.21%, Val Loss: 0.4754, Val Acc: 83.27%
Saved best model with Val Loss: 0.4279
Epoch 2/60 - Train Loss: 0.5858, Train Acc: 80.76%, Val Loss: 0.4279, Val Acc: 84.90%
Saved best model with Val Loss: 0.4151
Epoch 3/60 - Train Loss: 0.5333, Train Acc: 82.25%, Val Loss: 0.4151, Val Acc: 84.99%
Saved best model with Val Loss: 0.4062
Epoch 4/60 - Train Loss: 0.5076, Train Acc: 82.97%, Val Loss: 0.4062, Val Acc: 85.38%
Epoch 5/60 - Train Loss: 0.4907, Train Acc: 83.41%, Val Loss: 0.4141, Val Acc: 85.23%


KeyboardInterrupt: 